# ReCANet - PyTorch Implementation

Repeat Consumption-Aware Neural Network for Next Basket Recommendation (SIGIR 2022)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import warnings
warnings.filterwarnings('ignore')

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.models.recanet_torch import ReCANetTorch
from tecd_retail_recsys.metrics.recanet_metrics import recall_k, ndcg_k

print(f"System version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

System version: 3.11.14 (main, Oct 28 2025, 12:11:54) [Clang 20.1.4 ]
Pandas version: 2.3.3
Numpy version: 1.26.4
PyTorch version: 2.10.0
CUDA available: False


## Configuration

In [3]:
USER_EMBED_SIZE = 64
ITEM_EMBED_SIZE = 256
HISTORY_LEN = 50
BASKET_COUNT_MIN = 1
MIN_ITEM_COUNT = 1
JOB_ID = 1

EPOCHS = 10
BATCH_SIZE = 1024
LEARNING_RATE = 0.001

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

TOP_K = 100
EVAL_K_VALUES =[10, 100]

MODEL_DIR = './tecd_retail_recsys/models/recanet/trained'
DATA_DIR = './tecd_retail_recsys/models/recanet/data/tecd/'

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print(f"ReCANet Configuration:")
print(f"  User embedding size: {USER_EMBED_SIZE}")
print(f"  Item embedding size: {ITEM_EMBED_SIZE}")
print(f"  History length: {HISTORY_LEN}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Device: {DEVICE}")

ReCANet Configuration:
  User embedding size: 64
  Item embedding size: 256
  History length: 50
  Epochs: 10
  Batch size: 1024
  Learning rate: 0.001
  Device: cpu


## Data Preparation

In [4]:
dp = DataPreprocessor(
    day_begin=1082, 
    day_end=1308, 
    val_days=20, 
    test_days=20, 
    min_user_interactions=1, 
    min_item_interactions=20
)
train_df, val_df, test_df = dp.preprocess()

print(f"Train shape: {train_df.shape}")
print(f"Val shape: {val_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Number of users: {train_df['user_id'].nunique()}")
print(f"Number of items: {train_df['item_id'].nunique()}")

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 236,479,226 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (236479226, 12)
Filtered to 3,758,762 events with action_type='added-to-cart'
After filtering (min_user_interactions=1, min_item_interactions=20): 3,249,972 events, 84,944 users, 30,954 items
Created mappings: 84944 users, 30954 items
Temporal split - Train: days < 1269 (902,543 events), Val: days 1269-1288 (228,339 events), Test: days >= 1289 (223,395 events)
Users in each part (train, val, test) - 7425
Train shape: (902543, 12)
Val shape: (228339, 12)
Test shape: (223395, 12)
Number of users: 7425
Number of items: 30751


## Convert to Basket Format

In [ ]:
# корзины по 2 часа
def create_basket_id(df, window_hours=2):
    df = df.copy()

    if 'timestamp' in df.columns:
        df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
    else:
        df['datetime'] = df['timestamp']
    df['time_bucket'] = df['datetime'].dt.floor(f'{window_hours}H')
    df['basket_id'] = df.groupby(['user_id', 'time_bucket']).ngroup()
    df['date'] = df['datetime'].dt.date
    df = df[['date', 'basket_id', 'user_id', 'item_id', 'time_bucket']].drop_duplicates()
    return df

train_baskets = create_basket_id(train_df)
val_baskets = create_basket_id(val_df)
test_baskets = create_basket_id(test_df)

print(f"Train baskets: {train_baskets['basket_id'].nunique():,}")
print(f"Val baskets: {val_baskets['basket_id'].nunique():,}")
print(f"Test baskets: {test_baskets['basket_id'].nunique():,}")

In [ ]:
# train_baskets.to_csv(os.path.join(DATA_DIR, 'train_baskets_2h.csv'), index=False)
# val_baskets.to_csv(os.path.join(DATA_DIR, 'valid_baskets_2h.csv'), index=False)
# test_baskets.to_csv(os.path.join(DATA_DIR, 'test_baskets_2h.csv'), index=False)

In [8]:
train_baskets_loaded = pd.read_csv(os.path.join(DATA_DIR, 'train_baskets_2h.csv'))
valid_baskets_loaded = pd.read_csv(os.path.join(DATA_DIR, 'valid_baskets_2h.csv'))
test_baskets_loaded = pd.read_csv(os.path.join(DATA_DIR, 'test_baskets_2h.csv'))

print("Data loaded for ReCANet")
print(f"Train: {len(train_baskets_loaded):,} interactions")
print(f"Valid: {len(valid_baskets_loaded):,} interactions")
print(f"Test: {len(test_baskets_loaded):,} interactions")

Data loaded for ReCANet
Train: 897,840 interactions
Valid: 227,187 interactions
Test: 222,254 interactions


In [ ]:
model = ReCANetTorch(
    train_baskets_loaded, 
    test_baskets_loaded,
    valid_baskets_loaded,
    DATA_DIR,
    basket_count_min=BASKET_COUNT_MIN,
    min_item_count=MIN_ITEM_COUNT,
    user_embed_size=USER_EMBED_SIZE,
    item_embed_size=ITEM_EMBED_SIZE,
    h1=128,
    h2=128,
    h3=128,
    h4=128,
    h5=128,
    history_len=HISTORY_LEN,
    job_id=JOB_ID,
    device=DEVICE
)

print(f"\nReCANet initialized")
print(f"Number of test users: {len(model.test_users)}")
print(f"Number of items: {model.num_items}")
print(f"Number of users: {model.num_users}")
print(f"\nModel architecture:")
print(model.model)

Using device: cpu
Number of test users: 7425
Filtered items: 30751
Model initialized: 7426 users, 30752 items

ReCANet (PyTorch) initialized
Number of test users: 7425
Number of items: 30752
Number of users: 7426

Model architecture:
ReCANetModel(
  (user_embedding): Embedding(7426, 64)
  (item_embedding): Embedding(30752, 256)
  (dense1): Linear(in_features=320, out_features=128, bias=True)
  (relu1): ReLU()
  (dropout1): Dropout(p=0.2, inplace=False)
  (dense2): Linear(in_features=130, out_features=128, bias=True)
  (relu2): ReLU()
  (dropout2): Dropout(p=0.2, inplace=False)
  (lstm1): LSTM(128, 128, batch_first=True)
  (dropout_lstm1): Dropout(p=0.2, inplace=False)
  (lstm2): LSTM(128, 128, batch_first=True)
  (dropout_lstm2): Dropout(p=0.2, inplace=False)
  (dense3): Linear(in_features=128, out_features=128, bias=True)
  (relu3): ReLU()
  (dropout3): Dropout(p=0.2, inplace=False)
  (dense4): Linear(in_features=128, out_features=128, bias=True)
  (relu4): ReLU()
  (dropout4): Dropou

## Model Training

In [40]:
%%time
print("Starting model training...")
model.train(epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE)
print("\nModel training completed!")

Starting model training...
Loading cached training data from ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_50_train_cache.npz

Training data shape: (2996838, 50)
Positive examples: 19,544
Total examples: 2,996,838
Positive class weight: 152.34 (ratio: 0.65% positive)


Epoch 1/10: 100%|██████████| 2927/2927 [13:31<00:00,  3.61it/s, loss=1.1564, acc=67.93%]


Saved checkpoint: ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_1_64_256_128_128_128_128_128_50_epoch_1.pt


Epoch 2/10: 100%|██████████| 2927/2927 [13:28<00:00,  3.62it/s, loss=0.9873, acc=74.47%]


Saved checkpoint: ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_1_64_256_128_128_128_128_128_50_epoch_2.pt


...

Saved checkpoint: ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_1_64_256_128_128_128_128_128_50_epoch_8.pt


Epoch 9/10: 100%|██████████| 2927/2927 [13:35<00:00,  3.59it/s, loss=0.5409, acc=84.75%]


Saved checkpoint: ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_1_64_256_128_128_128_128_128_50_epoch_9.pt


Epoch 10/10: 100%|██████████| 2927/2927 [13:49<00:00,  3.53it/s, loss=0.5177, acc=85.47%]

Saved checkpoint: ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_1_64_256_128_128_128_128_128_50_epoch_10.pt

Training completed!

Model training completed!
CPU times: user 4h 17min 59s, sys: 36min 16s, total: 4h 54min 16s
Wall time: 2h 18min 30s


## Generate Predictions

In [41]:
%%time
print("Generating predictions...")
user_predictions = model.predict(batch_size=10_000)

print(f"\nPredictions generated for {len(user_predictions)} users")
first_user = list(user_predictions.keys())[0]
print(f"Sample - User {first_user}: {len(user_predictions[first_user])} items ranked")

Generating predictions...
Generating predictions...
Loading cached test data from ./tecd_retail_recsys/models/recanet/data/tecd/recanet_torch_50_test_cache.npz
Using last trained epoch: 10


Predicting: 100%|██████████| 64/64 [00:55<00:00,  1.16it/s]


Generated predictions for 7425 users

Predictions generated for 7425 users
Sample - User 11: 54 items ranked
CPU times: user 2min 40s, sys: 3.11 s, total: 2min 43s
Wall time: 56.3 s


## Evaluation

In [ ]:
user_val_baskets_df = valid_baskets_loaded.groupby('user_id')['item_id'].apply(list).reset_index()
user_val_baskets_dict = dict(zip(user_val_baskets_df['user_id'], user_val_baskets_df['item_id']))

final_users = set(model.test_users).intersection(set(user_val_baskets_dict.keys()))
print(f"Number of final val users: {len(final_users)}")

results = {}

for k in EVAL_K_VALUES:
    recall_scores = []
    ndcg_scores = []
    
    for user in final_users:
        pred_items = user_predictions.get(user, [])
        gt_items = user_val_baskets_dict[user]
        
        recall_scores.append(recall_k(gt_items, pred_items, k))
        ndcg_scores.append(ndcg_k(gt_items, pred_items, k))
    
    results[f'Recall@{k}'] = np.mean(recall_scores)
    results[f'NDCG@{k}'] = np.mean(ndcg_scores)
    
    print(f"\nk={k}:")
    print(f"  Recall@{k}: {results[f'Recall@{k}']:.4f}")
    print(f"  NDCG@{k}: {results[f'NDCG@{k}']:.4f}")

Number of final val users: 7425

k=10:
  Recall@10: 0.0572
  NDCG@10: 0.1743

k=100:
  Recall@100: 0.1701
  NDCG@100: 0.1552


## Save Results

In [ ]:
import json
from datetime import datetime

metadata = {
    'model': 'ReCANet-PyTorch',
    'timestamp': datetime.now().isoformat(),
    'config': {
        'user_embed_size': USER_EMBED_SIZE,
        'item_embed_size': ITEM_EMBED_SIZE,
        'history_len': HISTORY_LEN,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'learning_rate': LEARNING_RATE,
        'device': DEVICE
    },
    'data': {
        'train_users': int(train_baskets_loaded['user_id'].nunique()),
        'train_items': int(train_baskets_loaded['item_id'].nunique()),
        'test_users': len(final_users)
    },
    'results': {k: float(v) for k, v in results.items()}
}

metadata_path = os.path.join(MODEL_DIR, 'recanet_7_torch_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved metadata to {metadata_path}")

Saved metadata to ./tecd_retail_recsys/models/recanet/trained/recanet_7_torch_metadata.json


## ReCANet: эксперименты 
<table class="experiment-table">
    <thead>
        <tr>
            <th style="width: 3%">№</th>
            <th style="width: 15%">Название</th>
            <th style="width: 7%">user_emb</th>
            <th style="width: 7%">item_emb</th>
            <th style="width: 7%">history_len</th>
            <th style="width: 7%">epochs</th>
            <th style="width: 7%">h1-h5</th>
            <th style="width: 7%">dropout</th>
            <th style="width: 10%">basket_window</th>
            <th style="width: 9%">Recall@100</th>
            <th style="width: 9%">NDCG@100</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td class="num-cell">1</td>
            <td class="name-cell">Базовая конфигурация</td>
            <td class="param-cell">32</td>
            <td class="param-cell">128</td>
            <td class="param-cell">20</td>
            <td class="param-cell">10</td>
            <td class="param-cell">64</td>
            <td class="param-cell">0</td>
            <td class="param-cell">1 день</td>
            <td class="metric-cell">0.1750</td>
            <td class="metric-cell">0.1673</td>
        </tr>
        <tr>
            <td class="num-cell">2</td>
            <td class="name-cell">Увеличенные эмбеддинги</td>
            <td class="param-cell changed-param">64</td>
            <td class="param-cell changed-param">256</td>
            <td class="param-cell">20</td>
            <td class="param-cell">10</td>
            <td class="param-cell">64</td>
            <td class="param-cell">0</td>
            <td class="param-cell">1 день</td>
            <td class="metric-cell">0.1752</td>
            <td class="metric-cell">0.1635</td>
        </tr>
        <tr>
            <td class="num-cell">3</td>
            <td class="name-cell">Добавлен dropout</td>
            <td class="param-cell">64</td>
            <td class="param-cell">256</td>
            <td class="param-cell">20</td>
            <td class="param-cell">10</td>
            <td class="param-cell">64</td>
            <td class="param-cell changed-param">0.2</td>
            <td class="param-cell">1 день</td>
            <td class="metric-cell best-metric">0.1764</td>
            <td class="metric-cell best-metric">0.1737</td>
        </tr>
        <tr>
            <td class="num-cell">4</td>
            <td class="name-cell">Переход на реалистичные двухчасовые корзины</td>
            <td class="param-cell">64</td>
            <td class="param-cell">256</td>
            <td class="param-cell">20</td>
            <td class="param-cell">10</td>
            <td class="param-cell">64</td>
            <td class="param-cell">0.2</td>
            <td class="param-cell changed-param">2 часа</td>
            <td class="metric-cell">0.1712</td>
            <td class="metric-cell">0.1569</td>
        </tr>
        <tr>
            <td class="num-cell">5</td>
            <td class="name-cell">Увеличенная история</td>
            <td class="param-cell">64</td>
            <td class="param-cell">256</td>
            <td class="param-cell changed-param">50</td>
            <td class="param-cell">10</td>
            <td class="param-cell">64</td>
            <td class="param-cell">0.2</td>
            <td class="param-cell">2 часа</td>
            <td class="metric-cell">0.1713</td>
            <td class="metric-cell">0.1594</td>
        </tr>
        <tr>
            <td class="num-cell">6</td>
            <td class="name-cell">Больший item_emb</td>
            <td class="param-cell">64</td>
            <td class="param-cell changed-param">512</td>
            <td class="param-cell">50</td>
            <td class="param-cell changed-param">20</td>
            <td class="param-cell">64</td>
            <td class="param-cell">0.2</td>
            <td class="param-cell">2 часа</td>
            <td class="metric-cell">0.1712</td>
            <td class="metric-cell">0.1542</td>
        </tr>
        <tr>
            <td class="num-cell">7</td>
            <td class="name-cell">Большие скрытые слои</td>
            <td class="param-cell">64</td>
            <td class="param-cell">256</td>
            <td class="param-cell">50</td>
            <td class="param-cell">10</td>
            <td class="param-cell changed-param">128</td>
            <td class="param-cell">0.2</td>
            <td class="param-cell">2 часа</td>
            <td class="metric-cell">0.1701</td>
            <td class="metric-cell">0.1552</td>
        </tr>
    </tbody>
</table>

`Наилучшая конфигурация на двухчасовых корзинах смогла добиться NDCG@100 = 0.1594`

## Инференс лучшей модели

In [ ]:
best_model = ReCANetTorch(
    train_baskets_loaded, 
    test_baskets_loaded,
    valid_baskets_loaded,
    DATA_DIR,
    basket_count_min=BASKET_COUNT_MIN,
    min_item_count=MIN_ITEM_COUNT,
    user_embed_size=USER_EMBED_SIZE,
    item_embed_size=ITEM_EMBED_SIZE,
    h1=64,
    h2=64,
    h3=64,
    h4=64,
    h5=64,
    history_len=HISTORY_LEN,
    job_id=JOB_ID,
    device=DEVICE
)

checkpoint_path = 'tecd_retail_recsys/models/recanet/data/tecd/best_model.pt'
best_model.model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
best_model.model.eval()

In [33]:
from tecd_retail_recsys.metrics import calculate_metrics

user_predictions = best_model.predict(batch_size=10_000)
recs = pd.Series(user_predictions).reset_index().rename(columns={'index': 'user_id', 0: 'recanet_recs'})
joined = dp.get_grouped_data(train_df, val_df, test_df)
joined = joined.merge(recs, left_on='user_id', right_on='user_id')

recanet_metrics = calculate_metrics(joined, train_col='train_interactions', gt_col='val_interactions', model_preds='recanet_recs', verbose=True)

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (228339, 3) ratings_pred shape: (630733, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=11 gt_count=22 pred_count=54 overlap=6
  user_id=14 gt_count=5 pred_count=58 overlap=0
    [ID spaces] gt sample=[9341, 16732, 17585, 28024, 30789] range=[9341, 30789] | rec sample=[1809, 2175, 2811, 5687, 6061] range=[1809, 27978]
  user_id=21 gt_count=47 pred_count=58 overlap=15

At k=10:
  MAP@10       = 0.3131
  NDCG@10      = 0.6401
  Precision@10 = 0.2498
  Recall@10    = 0.0754

At k=100:
  MAP@100       = 0.1131
  NDCG@100      = 0.3253
  Precision@100 = 0.0761
  Recall@100    = 0.2069

Other Metrics:
  MRR                 = 0.3511
  Catalog Coverage    = 1.0000
  Diversity     = 0.9972  [0=same recs for all, 1=unique recs]
  Novelty             = 1.0000
  Serendipity   

`Модель ранжирует только айтемы из истории покупок пользователя (~50-100 айтемов) вместо всего каталога (~30,000 айтемов), поэтому видим такие высокие метрики, схожие с моделью TopPersonal.`